# Standard LoRA vs regularized LoRA

Compare the output-Gram ablation's standard LoRA and selected best regularized LoRA using the same prior samples and the same source and target images.

In [ ]:
from pathlib import Path
import csv
import sys

import matplotlib.pyplot as plt
import torch


REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))

from domain_knowledge_analysis import utils
from domain_knowledge_analysis.adapters import LoRAManager

## Settings

The checkpoints are resolved from the saved ablation results, so no model hash needs to be copied by hand.

In [ ]:
ABLATION_DIR = REPO_ROOT / "ablations/vae_mnist_rotated_mnist_output_gram"
NUM_GENERATIONS = 100
NUM_RECONSTRUCTIONS = 20
SEED = 42

BEST_MODELS = utils.load_config(ABLATION_DIR / "results/best_models.yaml")
BEST_REGULARIZED_MODEL = BEST_MODELS["models"][BEST_MODELS["qualitative_selection"]]

with open(ABLATION_DIR / "results/metrics.csv", newline="") as file:
    RESULTS = list(csv.DictReader(file))

STANDARD_LORA_MODEL = next(result for result in RESULTS if float(result["eta"]) == 0 and int(result["seed"]) == int(BEST_REGULARIZED_MODEL["seed"]))
STANDARD_CHECKPOINT = ABLATION_DIR / "models" / Path(STANDARD_LORA_MODEL["model_dir"]).name / "checkpoints/best.pt"
REGULARIZED_CHECKPOINT = ABLATION_DIR / "models" / Path(BEST_REGULARIZED_MODEL["model_dir"]).name / "checkpoints/best.pt"
REGULARIZED_LABEL = f"Best regularized LoRA\nη={float(BEST_REGULARIZED_MODEL['eta']):g}, M={int(BEST_REGULARIZED_MODEL['m'])}, s={float(BEST_REGULARIZED_MODEL['s']):g}"

## Load models and shared samples

In [ ]:
def load_lora_model(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    config = checkpoint["config"]

    if config.get("lora") is None:
        raise ValueError(f"Checkpoint is not a LoRA-adapted model: {checkpoint_path}")

    model = utils.create_model(config)
    lora_manager = LoRAManager(model, config)
    lora_manager.inject_adapters()
    model.load_state_dict(checkpoint["model_state_dict"])
    lora_manager.enable_adapters()
    return model.to(device).eval(), config


DEVICE = utils.get_device()
standard_model, standard_config = load_lora_model(STANDARD_CHECKPOINT, DEVICE)
regularized_model, regularized_config = load_lora_model(REGULARIZED_CHECKPOINT, DEVICE)

latent_dim = standard_config["model"]["encoder"]["latent_dim"]
if latent_dim != regularized_config["model"]["encoder"]["latent_dim"]:
    raise ValueError("The two models must have the same latent dimension.")

source_loader = utils.create_testing_dataloaders(standard_config)
target_loader = utils.create_testing_dataloaders(standard_config, standard_config["lora"]["transformed_dataset"])
source_images = next(iter(source_loader))[0][:NUM_RECONSTRUCTIONS]
target_images = next(iter(target_loader))[0][:NUM_RECONSTRUCTIONS]
latents = utils.sample_random_latents(NUM_GENERATIONS, latent_dim, utils.create_random_generator(SEED))

standard_generations = standard_model.generate_images(NUM_GENERATIONS, latents).cpu()
regularized_generations = regularized_model.generate_images(NUM_GENERATIONS, latents).cpu()
standard_source_reconstructions = standard_model.reconstruct_images(source_images).cpu()
regularized_source_reconstructions = regularized_model.reconstruct_images(source_images).cpu()
standard_target_reconstructions = standard_model.reconstruct_images(target_images).cpu()
regularized_target_reconstructions = regularized_model.reconstruct_images(target_images).cpu()

print(f"Standard LoRA: {STANDARD_CHECKPOINT}")
print(f"Regularized LoRA: {REGULARIZED_CHECKPOINT}")
print(f"Device: {DEVICE}")

In [ ]:
def show_image(axis, image):
    image = image.detach().cpu()

    if image.shape[0] == 1:
        axis.imshow(image[0], cmap="gray", vmin=0, vmax=1)
    else:
        axis.imshow(image.permute(1, 2, 0), vmin=0, vmax=1)

    axis.axis("off")


def plot_reconstructions(images, standard_reconstructions, regularized_reconstructions, title):
    columns = [(images, "Input"), (standard_reconstructions, "Standard LoRA"), (regularized_reconstructions, REGULARIZED_LABEL)]
    figure, axes = plt.subplots(len(images), len(columns), figsize=(7.5, 1.7 * len(images)), squeeze=False)

    for row in range(len(images)):
        for column, (column_images, label) in enumerate(columns):
            show_image(axes[row, column], column_images[row])

            if row == 0:
                axes[row, column].set_title(label)

    figure.suptitle(title)
    figure.tight_layout()
    plt.show()

## Prior generations

Each position uses exactly the same latent vector for both models.

In [ ]:
figure, axes = plt.subplots(10, 20, figsize=(20, 10), squeeze=False)

for index in range(NUM_GENERATIONS):
    row, column = divmod(index, 10)
    show_image(axes[row, column], standard_generations[index])
    show_image(axes[row, column + 10], regularized_generations[index])

figure.suptitle("100 generations from the same prior samples", y=0.995)
figure.text(0.25, 0.965, "Standard LoRA", ha="center", fontsize=14)
figure.text(0.75, 0.965, REGULARIZED_LABEL.replace("\n", " — "), ha="center", fontsize=14)
figure.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.94, wspace=0.05, hspace=0.05)
plt.show()

## Target reconstructions

In [ ]:
plot_reconstructions(target_images, standard_target_reconstructions, regularized_target_reconstructions, "Rotated MNIST reconstructions")

## Source reconstructions

In [ ]:
plot_reconstructions(source_images, standard_source_reconstructions, regularized_source_reconstructions, "Clean MNIST reconstructions")